# Telecom Customer Churn & Retention Intelligence
**Python • Pandas • Matplotlib • Jupyter**

Business question: **Which behavioural and service signals are most strongly associated with customer churn, and which customer groups should the telecom company prioritise for retention?**

## 1. Setup and data source

The project uses the UCI Iranian Churn dataset. `Charge Amount` is ordinal rather than currency, so this analysis does not invent a revenue-at-risk metric.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
RAW = Path('../02_data/raw/Customer_Churn.csv')
CLEAN = Path('../02_data/cleaned/telecom_churn_cleaned.csv')
VIS = Path('../03_visuals')
OUT = Path('../04_outputs')
VIS.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)


In [2]:
df = pd.read_csv(RAW)
df.columns = (df.columns.str.strip().str.replace(r'\\s+', '_', regex=True).str.lower())
df.insert(0, 'record_id', range(1, len(df)+1))
print(f'Rows: {len(df):,}')
print(f'Columns: {df.shape[1]-1}')
df.head()


Rows: 3150\nColumns: 16\n

## 2. Data quality checks

In [3]:
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicate source rows:', int(df.drop(columns='record_id').duplicated().sum()))
# Duplicates are retained because the source has no customer identifier; identical aggregated profiles may represent different customers.


Missing values: 0\nDuplicate source rows: 300\n

Identical feature rows are **not automatically removed**. Without a source customer identifier, identical aggregated profiles cannot safely be assumed to be duplicate customers.

In [4]:
df['complaint_label'] = df['complains'].map({0:'No complaint', 1:'Complaint'})
df['tariff_label'] = df['tariff_plan'].map({1:'Pay as you go', 2:'Contractual'})
df['status_label'] = df['status'].map({1:'Active', 2:'Non-active'})
df['churn_label'] = df['churn'].map({0:'Retained', 1:'Churned'})
df['tenure_band'] = pd.cut(df['subscription_length'], [-1,12,24,36,float('inf')], labels=['0–12 months','13–24 months','25–36 months','37+ months'])
df['call_activity_band'] = pd.cut(df['frequency_of_use'], [-1,0,25,75,float('inf')], labels=['No calls','1–25 calls','26–75 calls','76+ calls'])
df['sms_activity_band'] = pd.cut(df['frequency_of_sms'], [-1,0,25,100,float('inf')], labels=['No SMS','1–25 SMS','26–100 SMS','101+ SMS'])


## 3. Overall churn

In [5]:
customers = len(df)
churners = int(df['churn'].sum())
churn_rate = df['churn'].mean() * 100
print(f'Customers: {customers:,}')
print(f'Churners: {churners:,}')
print(f'Churn rate: {churn_rate:.1f}%')


Customers: 3,150\nChurners: 495\nChurn rate: 15.7%\n

## 4. Segment churn rates

In [6]:
def churn_summary(column):
    return (df.groupby(column, observed=False)
              .agg(customers=('churn','size'), churners=('churn','sum'), avg_customer_value=('customer_value','mean'))
              .assign(churn_rate=lambda x: x['churners']/x['customers']*100)
              .reset_index())

complaint_summary = churn_summary('complaint_label')
status_summary = churn_summary('status_label')
tariff_summary = churn_summary('tariff_label')
tenure_summary = churn_summary('tenure_band')
call_summary = churn_summary('call_activity_band')
sms_summary = churn_summary('sms_activity_band')


In [7]:
print(f"Complaint churn: {complaint_summary.loc[complaint_summary['complaint_label']=='Complaint','churn_rate'].iloc[0]:.1f}%")
print(f"No-complaint churn: {complaint_summary.loc[complaint_summary['complaint_label']=='No complaint','churn_rate'].iloc[0]:.1f}%")
print(f"Non-active churn: {status_summary.loc[status_summary['status_label']=='Non-active','churn_rate'].iloc[0]:.1f}%")
print(f"Active churn: {status_summary.loc[status_summary['status_label']=='Active','churn_rate'].iloc[0]:.1f}%")


Complaint churn: 83%\nNo-complaint churn: 10.1%\nNon-active churn: 47.3%\nActive churn: 5.3%\n

## 5. Visual analysis

In [ ]:
def plot_churn(summary, category, order, title, filename):
    plot_df = summary.set_index(category).reindex(order).reset_index()
    fig, ax = plt.subplots(figsize=(9,5))
    ax.barh(plot_df[category].astype(str), plot_df['churn_rate'])
    ax.set_title(title, loc='left', fontweight='bold')
    ax.set_xlabel('Churn rate (%)')
    ax.invert_yaxis()
    for i,v in enumerate(plot_df['churn_rate']): ax.text(v+0.8, i, f'{v:.1f}%', va='center')
    fig.tight_layout()
    fig.savefig(VIS/filename, dpi=160, bbox_inches='tight')
    plt.show()

plot_churn(complaint_summary,'complaint_label',['Complaint','No complaint'],'Churn by complaint behaviour','complaint-vs-churn.png')
plot_churn(call_summary,'call_activity_band',['No calls','1–25 calls','26–75 calls','76+ calls'],'Churn by call activity','call-activity-vs-churn.png')
plot_churn(tenure_summary,'tenure_band',['0–12 months','13–24 months','25–36 months','37+ months'],'Churn by subscription length','tenure-vs-churn.png')


## 6. Transparent retention-priority heuristic

This is deliberately a **business-rule prioritisation score**, not a machine-learning probability.

In [8]:
df['retention_priority_score'] = (
    (df['complains'].eq(1))*3 +
    (df['status'].eq(2))*3 +
    (df['frequency_of_use'].le(25))*2 +
    (df['frequency_of_sms'].le(25))*1 +
    (df['subscription_length'].le(12))*1 +
    (df['tariff_plan'].eq(1))*1
)
df['retention_priority'] = pd.cut(df['retention_priority_score'], [-1,3,6,float('inf')], labels=['Low','Medium','High'])
priority_summary = churn_summary('retention_priority')
priority_summary


In [9]:
for band in ['High','Medium','Low']:
    row = priority_summary.loc[priority_summary['retention_priority'].astype(str).eq(band)].iloc[0]
    print(f"{band}: {int(row['customers'])} customers, {row['churn_rate']:.1f}% churn")


High: 549 customers, 56.6% churn\nMedium: 522 customers, 28% churn\nLow: 2079 customers, 1.8% churn\n

## 7. Export reproducible outputs

In [10]:
df.to_csv(CLEAN, index=False)
priority_summary.to_csv(OUT/'retention_priority_segments.csv', index=False)
df.loc[df['retention_priority'].astype(str).eq('High')].to_csv(OUT/'high_priority_records.csv', index=False)


## 8. Decision-support conclusion

The analysis shows a consistent pattern: churn is concentrated among customers showing **service friction and weak engagement**. Complaint customers churn at **83%**, non-active customers at **47.3%**, and customers with no calls at **52.6%**. By contrast, highly engaged customers show substantially lower churn.

The priority score is useful as an operational triage layer, but it should be validated on future outcomes before being used for automated retention decisions.